In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder.getOrCreate()

test_results = []

# TEST FRAMEWORK

def run_test(test_name, condition, failure_reason=None):

    status = "PASSED" if condition else "FAILED"

    print(f"{status}: {test_name}")

    test_results.append({
        "test_name": test_name,
        "status": status,
        "failure_reason": failure_reason
    })

    if not condition:

        raise Exception(f"{test_name} FAILED")

# BRONZE TESTS

bronze_df = spark.table(
    "realtime_weather.bronze.bronze_weather_data"
)

run_test(
    "Bronze table has data",
    bronze_df.count() > 0
)

run_test(
    "No null raw_response",
    bronze_df.filter(
        "raw_response IS NULL"
    ).count() == 0
)


# SILVER TESTS

silver_df = spark.table(
    "realtime_weather.silver.silver_weather_clean"
)

run_test(
    "Silver table has data",
    silver_df.count() > 0
)

invalid_temp = silver_df.filter("""
temperature_celsius > 60
OR temperature_celsius < -50
""").count()

run_test(
    "Temperature validation",
    invalid_temp == 0,
    "Invalid temperature found"
)

invalid_humidity = silver_df.filter("""
humidity_percent > 100
OR humidity_percent < 0
""").count()

run_test(
    "Humidity validation",
    invalid_humidity == 0,
    "Invalid humidity found"
)

# GOLD TESTS


gold_df = spark.table(
    "realtime_weather.gold.gold_city_current"
)

run_test(
    "One row per city",
    gold_df.count()
    ==
    gold_df.select("city").distinct().count(),
    "Duplicate city rows found"
)


hourly_df = spark.table(
    "realtime_weather.gold.gold_hourly_summary"
)

run_test(
    "Hourly summary populated",
    hourly_df.count() > 0
)


api_df = spark.table(
    "realtime_weather.gold.gold_api_health"
)

invalid_success = api_df.filter("""
success_rate < 0
OR success_rate > 100
""").count()

run_test(
    "Success rate valid",
    invalid_success == 0,
    "Invalid success_rate found"
)


# PIPELINE FRESHNESS TEST


bronze_ts = spark.sql("""
SELECT MAX(call_timestamp) ts
FROM realtime_weather.bronze.bronze_weather_data
""").collect()[0]["ts"]

silver_ts = spark.sql("""
SELECT MAX(ingestion_timestamp) ts
FROM realtime_weather.silver.silver_weather_clean
""").collect()[0]["ts"]

run_test(
    "Silver updated after Bronze",
    silver_ts >= bronze_ts,
    "Silver pipeline stale"
)

# STORE TEST RESULTS


results_df = spark.createDataFrame(test_results)

results_df = results_df.withColumn(
    "execution_time",
    current_timestamp()
)

results_df.write.format("delta") \
    .mode("append") \
    .saveAsTable(
        "realtime_weather.monitoring.pipeline_test_results"
    )

display(results_df)

print(" All Pipeline Testing Complete")

PASSED: Bronze table has data
PASSED: No null raw_response
PASSED: Silver table has data
PASSED: Temperature validation
PASSED: Humidity validation
PASSED: One row per city
PASSED: Hourly summary populated
PASSED: Success rate valid
PASSED: Silver updated after Bronze


failure_reason,status,test_name,execution_time
null,PASSED,Bronze table has data,2026-05-10T17:29:37.872Z
null,PASSED,No null raw_response,2026-05-10T17:29:37.872Z
null,PASSED,Silver table has data,2026-05-10T17:29:37.872Z
Invalid temperature found,PASSED,Temperature validation,2026-05-10T17:29:37.872Z
Invalid humidity found,PASSED,Humidity validation,2026-05-10T17:29:37.872Z
Duplicate city rows found,PASSED,One row per city,2026-05-10T17:29:37.872Z
null,PASSED,Hourly summary populated,2026-05-10T17:29:37.872Z
Invalid success_rate found,PASSED,Success rate valid,2026-05-10T17:29:37.872Z
Silver pipeline stale,PASSED,Silver updated after Bronze,2026-05-10T17:29:37.872Z


 All Pipeline Testing Complete


In [0]:
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import *
# from datetime import timedelta

# spark = SparkSession.builder.getOrCreate()

# test_results = []

# # TEST FRAMEWORK

# def run_test(test_name, passed, details):

#     status = "PASSED" if passed else "FAILED"

#     print(f"{status}: {test_name}")

#     test_results.append({
#         "test_name": test_name,
#         "status": status,
#         "details": details
#     })

#     if not passed:
#         raise Exception(f"{test_name} FAILED")

# # LOAD DATA

# silver_df = spark.table(
#     "realtime_weather.silver.silver_weather_clean"
# )

# gold_df = spark.table(
#     "realtime_weather.gold.gold_city_current"
# )

# bronze_df = spark.table(
#     "realtime_weather.bronze.bronze_weather_data"
# )

# # DYNAMIC ROW COUNT DROP DETECTION


# today_count = silver_df.filter(
#     col("event_date") == current_date()
# ).count()

# yesterday_count = silver_df.filter(
#     col("event_date") == date_sub(current_date(), 1)
# ).count()

# # adaptive threshold
# minimum_expected = yesterday_count * 0.5

# run_test(
#     "Dynamic row count validation",
#     today_count >= minimum_expected,
#     f"today={today_count}, yesterday={yesterday_count}"
# )

# # TEMPERATURE DISTRIBUTION VALIDATION

# extreme_temp_count = silver_df.filter("""
# temperature_celsius > 55
# OR temperature_celsius < -20
# """).count()

# extreme_temp_ratio = extreme_temp_count / silver_df.count()

# run_test(
#     "Temperature distribution validation",
#     extreme_temp_ratio < 0.1,
#     f"extreme_ratio={extreme_temp_ratio}"
# )

# # API LATENCY VALIDATION

# high_latency_count = bronze_df.filter("""
# response_latency_ms > 10000
# """).count()

# latency_ratio = high_latency_count / bronze_df.count()

# run_test(
#     "API latency validation",
#     latency_ratio < 0.2,
#     f"high_latency_ratio={latency_ratio}"
# )


# # DYNAMIC DUPLICATE DETECTION


# duplicate_ratio = gold_df.count() / \
# gold_df.select("city").distinct().count()

# run_test(
#     "Dynamic duplicate ratio validation",
#     duplicate_ratio == 1,
#     f"ratio={duplicate_ratio}"
# )


# # DYNAMIC PIPELINE FRESHNESS


# latest_bronze = bronze_df.select(
#     max("call_timestamp").alias("ts")
# ).collect()[0]["ts"]

# latest_silver = silver_df.select(
#     max("ingestion_timestamp").alias("ts")
# ).collect()[0]["ts"]

# time_diff = (
#     latest_silver - latest_bronze
# ).total_seconds()

# run_test(
#     "Dynamic freshness validation",
#     time_diff < 3600,
#     f"delay_seconds={time_diff}"
# )


# # DYNAMIC CITY COVERAGE VALIDATION


# bronze_cities = bronze_df.select(
#     "city_requested"
# ).distinct().count()

# gold_cities = gold_df.select(
#     "city"
# ).distinct().count()

# coverage_ratio = gold_cities / bronze_cities

# run_test(
#     "Dynamic city coverage validation",
#     coverage_ratio >= 0.9,
#     f"coverage_ratio={coverage_ratio}"
# )

# # TEST 7
# # DYNAMIC SEVERITY DISTRIBUTION

# severity_distribution = silver_df.groupBy(
#     "weather_severity_score"
# ).count()

# extreme_count = severity_distribution.filter(
#     col("weather_severity_score") >= 6
# ).agg(sum("count")).collect()[0][0]

# if extreme_count is None:
#     extreme_count = 0

# total_rows = silver_df.count()

# extreme_ratio = extreme_count / total_rows

# run_test(
#     "Dynamic severity distribution validation",
#     extreme_ratio < 0.5,
#     f"extreme_ratio={extreme_ratio}"
# )

# # STORE RESULTS

# results_df = spark.createDataFrame(test_results)

# results_df = results_df.withColumn(
#     "execution_time",
#     current_timestamp()
# )

# results_df.write.format("delta") \
#     .mode("append") \
#     .saveAsTable(
#         "realtime_weather.monitoring.pipeline_test_results"
#     )

# display(results_df)

# print(" ALL DYNAMIC TESTS PASSED")